# SASRec 训练（Amazon Beauty）

自包含版本，无需 git clone。

**流程：**
1. 确认 GPU
2. 上传 `beauty_data.pkl`
3. 训练（约 200 epochs，T4 上每 epoch ~20s）
4. 下载 checkpoint

**参考指标（TIGER 论文，all-rank）：** SASRec Recall@10 = 0.0605，NDCG@10 = 0.0318

In [ ]:
# Cell 1：确认 GPU
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  未检测到 GPU，请到 Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：上传 beauty_data.pkl
# 从本地上传（在项目 data/ 目录下）
import os
from google.colab import files

os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('data/beauty_data.pkl'):
    print('请上传 beauty_data.pkl ...')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, 'data/beauty_data.pkl')
    print('上传完成')
else:
    print('data/beauty_data.pkl 已存在，跳过上传')

In [ ]:
# Cell 3：SASRec 模型定义
import torch
import torch.nn as nn

class SASRec(nn.Module):
    def __init__(self, num_items, hidden_size=64, num_layers=2,
                 num_heads=1, dropout=0.2, maxlen=50):
        super().__init__()
        self.hidden_size = hidden_size
        self.maxlen      = maxlen

        # item embedding：index 0 留给 padding
        self.item_emb = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.pos_emb  = nn.Embedding(maxlen, hidden_size)

        self.emb_dropout = nn.Dropout(dropout)
        self.layer_norm  = nn.LayerNorm(hidden_size, eps=1e-8)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, item_seq):
        B, L = item_seq.shape
        positions = torch.arange(L, device=item_seq.device).unsqueeze(0).expand(B, -1)
        x = self.item_emb(item_seq) + self.pos_emb(positions)
        x = self.emb_dropout(self.layer_norm(x))

        causal_mask = torch.triu(
            torch.ones(L, L, device=item_seq.device, dtype=torch.bool), diagonal=1)
        pad_mask = (item_seq == 0)

        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=pad_mask,
                             is_causal=True)
        return x[:, -1, :]  # 最后一个位置作为序列表示

    def predict(self, item_seq, candidate_ids):
        seq_emb  = self.forward(item_seq)                                          # (B, H)
        cand_emb = self.item_emb(candidate_ids)                                    # (B, C, H)
        scores   = torch.bmm(cand_emb, seq_emb.unsqueeze(-1)).squeeze(-1)          # (B, C)
        return scores

print('SASRec 定义完成')

In [ ]:
# Cell 4：Dataset / DataLoader / 评估函数
import math
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

CONFIG = {
    'maxlen':      50,
    'hidden_size': 64,
    'num_layers':  2,
    'num_heads':   1,
    'dropout':     0.2,
    'lr':          1e-3,
    'weight_decay': 1e-4,
    'batch_size':  256,
    'num_epochs':  200,
    'patience':    20,   # early stopping（每 val_every epoch 评估一次）
    'val_every':   10,
    'num_neg':     1,    # BPR：每个正样本对应的负样本数
}
K_LIST = [5, 10]


class SASRecDataset(Dataset):
    def __init__(self, user_seqs, num_items, maxlen=50, num_neg=1):
        self.samples   = []
        self.num_items = num_items
        self.num_neg   = num_neg

        for user, seq in user_seqs.items():
            if len(seq) < 2:          # 空历史会导致 all-masked attention → NaN
                continue
            interacted = set(seq)
            for i in range(1, len(seq)):
                input_seq = seq[:i][-maxlen:]
                pos_item  = seq[i]
                self.samples.append((input_seq, pos_item, interacted))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        input_seq, pos_item, interacted = self.samples[idx]
        neg_items = []
        while len(neg_items) < self.num_neg:
            neg = random.randint(1, self.num_items)
            if neg not in interacted:
                neg_items.append(neg)
        return input_seq, pos_item, neg_items[0]


def collate_sasrec(batch):
    maxlen    = max(max(len(s) for s, _, _ in batch), 1)
    padded    = [[0] * (maxlen - len(s)) + list(s) for s, _, _ in batch]
    input_ids = torch.tensor(padded, dtype=torch.long)
    pos_items = torch.tensor([p for _, p, _ in batch], dtype=torch.long)
    neg_items = torch.tensor([n for _, _, n in batch], dtype=torch.long)
    return input_ids, pos_items, neg_items


def evaluate_allrank(model, test_seqs, num_items, device, maxlen=50, k_list=K_LIST):
    model.eval()
    metrics      = {k: {'Recall': [], 'NDCG': []} for k in k_list}
    max_k        = max(k_list)
    all_item_ids = torch.arange(1, num_items + 1, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for user, full_seq in tqdm(test_seqs.items(), desc='Evaluating (all-rank)', leave=False):
            target   = full_seq[-1]
            history  = full_seq[:-1][-maxlen:]
            pad_len  = maxlen - len(history)
            input_ids = torch.tensor([[0] * pad_len + list(history)],
                                     dtype=torch.long, device=device)

            scores     = model.predict(input_ids, all_item_ids).squeeze(0)
            _, topk_idx = torch.topk(scores, max_k)
            topk_items  = (topk_idx + 1).tolist()

            for k in k_list:
                topk = topk_items[:k]
                if target in topk:
                    rank = topk.index(target) + 1
                    metrics[k]['Recall'].append(1)
                    metrics[k]['NDCG'].append(1 / math.log2(rank + 1))
                else:
                    metrics[k]['Recall'].append(0)
                    metrics[k]['NDCG'].append(0.0)

    model.train()
    return {k: {m: float(np.mean(v)) for m, v in mv.items()} for k, mv in metrics.items()}


print('Dataset / 评估函数定义完成')

In [ ]:
# Cell 5：训练
import pickle

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

data       = pickle.load(open('data/beauty_data.pkl', 'rb'))
num_items  = len(data['item2id'])
train_seqs = data['train']
val_seqs   = data['val']
test_seqs  = data['test']
print(f'item 数: {num_items}，训练用户数: {len(train_seqs)}')

train_dataset = SASRecDataset(train_seqs, num_items, CONFIG['maxlen'], CONFIG['num_neg'])
train_loader  = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                           shuffle=True, collate_fn=collate_sasrec, num_workers=2)
print(f'训练样本数: {len(train_dataset)}')

model = SASRec(
    num_items   = num_items,
    hidden_size = CONFIG['hidden_size'],
    num_layers  = CONFIG['num_layers'],
    num_heads   = CONFIG['num_heads'],
    dropout     = CONFIG['dropout'],
    maxlen      = CONFIG['maxlen'],
).to(device)
print(f'参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}M')

optimizer = torch.optim.Adam(model.parameters(),
                             lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
bpr_loss  = lambda pos_s, neg_s: -torch.log(torch.sigmoid(pos_s - neg_s)).mean()

best_recall10  = 0.0
patience_count = 0
print(f'\n开始训练，共 {CONFIG["num_epochs"]} epochs\n')

for epoch in range(1, CONFIG['num_epochs'] + 1):
    total_loss = 0.0
    for input_ids, pos_items, neg_items in train_loader:
        input_ids = input_ids.to(device)
        pos_items = pos_items.to(device)
        neg_items = neg_items.to(device)

        candidates = torch.stack([pos_items, neg_items], dim=1)
        scores     = model.predict(input_ids, candidates)
        loss       = bpr_loss(scores[:, 0], scores[:, 1])

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    if epoch % CONFIG['val_every'] == 0 or epoch == 1:
        summary  = evaluate_allrank(model, val_seqs, num_items, device, CONFIG['maxlen'])
        recall10 = summary[10]['Recall']
        print(f'Epoch {epoch:3d}  loss={avg_loss:.4f}  '
              f'Recall@5={summary[5]["Recall"]:.4f}  NDCG@5={summary[5]["NDCG"]:.4f}  '
              f'Recall@10={recall10:.4f}  NDCG@10={summary[10]["NDCG"]:.4f}')

        if recall10 > best_recall10:
            best_recall10  = recall10
            patience_count = 0
            torch.save(model.state_dict(), 'checkpoints/sasrec_best.pt')
            print(f'  ✓ 保存最优模型 (Recall@10={best_recall10:.4f})')
        else:
            patience_count += 1
            if patience_count >= CONFIG['patience']:
                print(f'\nEarly stopping at epoch {epoch}')
                break
    else:
        print(f'Epoch {epoch:3d}  loss={avg_loss:.4f}')

print(f'\n训练结束，best val Recall@10 = {best_recall10:.4f}')

In [ ]:
# Cell 6：测试集最终评估
model.load_state_dict(torch.load('checkpoints/sasrec_best.pt',
                                 map_location=device, weights_only=True))
final = evaluate_allrank(model, test_seqs, num_items, device, CONFIG['maxlen'])

print('=' * 55)
print('  SASRec 测试集结果（all-rank）')
print('=' * 55)
for k in K_LIST:
    print(f'  Recall@{k:<2} = {final[k]["Recall"]:.4f}   '
          f'NDCG@{k:<2} = {final[k]["NDCG"]:.4f}')
print('─' * 55)
print('  TIGER 论文 SASRec: Recall@10=0.0605  NDCG@10=0.0318')
print('=' * 55)

In [ ]:
# Cell 7：下载 checkpoint
from google.colab import files
files.download('checkpoints/sasrec_best.pt')
size_mb = os.path.getsize('checkpoints/sasrec_best.pt') / 1e6
print(f'已下载 sasrec_best.pt ({size_mb:.1f} MB)')